# Player Stats Extraction

In [ ]:
# Library Imports
import pandas as pd
import time
import numpy as np
from datetime import datetime

# Web scraping libraries
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Error handling
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

# Data handling
import json
import os

In [7]:
# Load player data and setup browser for stats extraction
print("Loading NBA player data...")

# Load the complete player database
df_players = pd.read_csv('../data/pages/all_nba_players.csv')

print(f"Total players loaded: {len(df_players):,}")
print(f"Sample of loaded data:")
print(df_players.head())

Loading NBA player data...
Total players loaded: 5,313
Sample of loaded data:
           player_name                                         player_url  \
0       Alaa Abdelnaby  https://www.basketball-reference.com/players/a...   
1      Zaid Abdul-Aziz  https://www.basketball-reference.com/players/a...   
2  Kareem Abdul-Jabbar  https://www.basketball-reference.com/players/a...   
3   Mahmoud Abdul-Rauf  https://www.basketball-reference.com/players/a...   
4    Tariq Abdul-Wahad  https://www.basketball-reference.com/players/a...   

  letter  
0      A  
1      A  
2      A  
3      A  
4      A  


In [6]:

# Setup Chrome browser for stats extraction
print("\nSetting up Chrome browser...")
options = Options()
options.add_argument('--headless')  # Run in background
options.add_argument('--no-sandbox')  # Bypass OS security
options.add_argument('--disable-dev-shm-usage')  # Use regular storage instead of in-memory storage

service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

print("Browser setup complete")
print(f"Ready to extract stats from {len(df_players)} player pages")


Setting up Chrome browser...
Browser setup complete
Ready to extract stats from 5313 player pages


In [13]:
# Find the row for Kawhi Leonard
kawhi_row = df_players[df_players['player_name'] == 'Kawhi Leonard']

# Access the 'player_url' from that row
# .values[0] is used to get the first value from the column
kawhi_url = kawhi_row['player_url'].values[0]

print(kawhi_url)

https://www.basketball-reference.com/players/l/leonaka01.html


In [16]:
# Navigate to Kawhi's page
driver.get(kawhi_url)

# Wait for the advanced stats table to load
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "advanced"))
    )
    print("Advanced stats table loaded successfully")

    # Get page source and parse with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Find the advanced stats table
    advanced_table = soup.find('table', id='advanced')

    if advanced_table:
        print("Advanced stats table found")

        # Get table headers to understand structure
        headers = advanced_table.find('thead').find_all('th')
        header_names = [th.get('data-stat', th.text.strip()) for th in headers]

        print(f"Table headers found: {len(header_names)}")
        print(f"Header names: {header_names[:10]}...")  # Show first 10

    else:
        print("Advanced stats table not found")

except TimeoutException:
    print("Timeout waiting for advanced stats table")
    print(f"Current URL: {driver.current_url}")

Advanced stats table loaded successfully
Advanced stats table found
Table headers found: 29
Header names: ['year_id', 'age', 'team_name_abbr', 'comp_name_abbr', 'pos', 'games', 'games_started', 'mp', 'per', 'ts_pct']...


In [19]:
# Extract Kawhi's season-by-season stats
print("Extracting Kawhi's season stats...")

# Get all data rows from the advanced table
rows = advanced_table.find('tbody').find_all('tr')

# Filter out header rows and totals rows
data_rows = []
for row in rows:
    # Skip rows that are headers or don't have season data
    season_cell = row.find('th', {'data-stat': 'year_id'})
    if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
        data_rows.append(row)

print(f"Found {len(data_rows)} seasons of data")

# Extract the specific columns we need
kawhi_seasons = []

target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'usg_pct', 'ws', 'ws_per_48']

for row in data_rows:
    season_data = {}
    
    # Extract each target stat
    for stat in target_stats:
        cell = row.find(['th', 'td'], {'data-stat': stat})
        if cell:
            value = cell.text.strip()
            season_data[stat] = value if value else None
        else:
            season_data[stat] = None
    
    kawhi_seasons.append(season_data)

# Convert to DataFrame for easy viewing
kawhi_df = pd.DataFrame(kawhi_seasons)

print("\nKawhi Leonard's extracted stats:")
print(kawhi_df)

# Check for any missing key stats
print(f"\nData completeness check:")
for stat in ['per', 'bpm', 'ws', 'ws_per_48']:
    missing = kawhi_df[stat].isna().sum()
    total = len(kawhi_df)
    print(f"{stat.upper()}: {total - missing}/{total} seasons available")

Extracting Kawhi's season stats...
Found 14 seasons of data

Kawhi Leonard's extracted stats:
    year_id   age         team_name_abbr games    mp   per   bpm  vorp  \
0   2011-12    20                    SAS    64  1534  16.6   3.4   2.1   
1   2012-13    21                    SAS    58  1810  16.4   3.1   2.3   
2   2013-14    22                    SAS    66  1923  19.4   5.1   3.5   
3   2014-15    23                    SAS    64  2033  22.0   5.9   4.1   
4   2015-16    24                    SAS    72  2380  26.0   9.1   6.7   
5   2016-17    25                    SAS    74  2474  27.6   9.4   7.1   
6   2017-18    26                    SAS     9   210  26.0  10.1   0.6   
7   2018-19    27                    TOR    60  2040  25.8   7.2   4.7   
8   2019-20    28                    LAC    57  1848  26.9   8.9   5.1   
9   2020-21    29                    LAC    52  1773  26.0   7.3   4.2   
10  2021-22  None  Did not play - injury  None  None  None  None  None   
11  2022-23    31 

In [20]:
# Extract stats from 20 random players to test our pipeline
print("Testing extraction pipeline with 20 random players...")

# Select 20 random players from our database
random_players = df_players.sample(n=20, random_state=42)
print(f"Selected {len(random_players)} random players for testing")

# Define our target stats
target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'ws', 'ws_per_48']

# Store all extracted data
all_player_stats = []
successful_extractions = 0
failed_extractions = 0

for idx, (_, player) in enumerate(random_players.iterrows(), 1):
    player_name = player['player_name']
    player_url = player['player_url']
    
    print(f"\n[{idx}/20] Extracting: {player_name}")
    
    try:
        # Navigate to player page
        driver.get(player_url)
        
        # Wait for advanced stats table
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "advanced"))
        )
        
        # Parse the page
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        advanced_table = soup.find('table', id='advanced')
        
        if advanced_table:
            # Get data rows (exclude headers and totals)
            rows = advanced_table.find('tbody').find_all('tr')
            data_rows = []
            
            for row in rows:
                season_cell = row.find('th', {'data-stat': 'year_id'})
                if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
                    data_rows.append(row)
            
            # Extract stats for each season
            for row in data_rows:
                season_data = {'player_name': player_name}
                
                for stat in target_stats:
                    cell = row.find(['th', 'td'], {'data-stat': stat})
                    if cell:
                        value = cell.text.strip()
                        season_data[stat] = value if value else None
                    else:
                        season_data[stat] = None
                
                all_player_stats.append(season_data)
            
            print(f"  ✓ Extracted {len(data_rows)} seasons")
            successful_extractions += 1
            
        else:
            print(f"  ✗ No advanced stats table found")
            failed_extractions += 1
    
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        failed_extractions += 1
    
    # Rate limiting - 3 second delay
    time.sleep(3)

# Convert to DataFrame
df_extracted = pd.DataFrame(all_player_stats)

print(f"\n=== EXTRACTION SUMMARY ===")
print(f"Successful players: {successful_extractions}/20")
print(f"Failed players: {failed_extractions}/20")
print(f"Total seasons extracted: {len(df_extracted)}")

if len(df_extracted) > 0:
    print(f"\nSample of extracted data:")
    print(df_extracted.head())
    
    # Save test data
    df_extracted.to_csv('../data/pages/test_player_stats_20.csv', index=False)
    print(f"\nSaved test data to '../data/pages/test_player_stats_20.csv'")
else:
    print("\nNo data extracted - check for errors above")

Testing extraction pipeline with 20 random players...
Selected 20 random players for testing

[1/20] Extracting: Ed Dahler
  ✓ Extracted 2 seasons

[2/20] Extracting: Zach Randolph
  ✓ Extracted 19 seasons

[3/20] Extracting: Jim Kissane
  ✓ Extracted 1 seasons

[4/20] Extracting: Andrés Guibert
  ✓ Extracted 2 seasons

[5/20] Extracting: Chris Gatling
  ✓ Extracted 20 seasons

[6/20] Extracting: Dan Gadzuric
  ✓ Extracted 12 seasons

[7/20] Extracting: Ed Beach
  ✓ Extracted 3 seasons

[8/20] Extracting: Phil Wagner
  ✓ Extracted 1 seasons

[9/20] Extracting: Danny Manning
  ✓ Extracted 17 seasons

[10/20] Extracting: Dave Hoppen
  ✓ Extracted 11 seasons

[11/20] Extracting: Mouhamed Gueye
  ✓ Extracted 2 seasons

[12/20] Extracting: Darius Johnson-Odom
  ✓ Extracted 2 seasons

[13/20] Extracting: Leroy Combs
  ✓ Extracted 1 seasons

[14/20] Extracting: Amile Jefferson
  ✓ Extracted 2 seasons

[15/20] Extracting: Jerry Pender
  ✓ Extracted 2 seasons

[16/20] Extracting: Reggie Royals


In [21]:
# Full dataset extraction - All NBA players
print("Starting full dataset extraction...")
print(f"Total players to process: {len(df_players)}")

# Estimate time based on actual performance (20 players = 3m 54.6s)
actual_rate = 234.6 / 20  # seconds per player from test run
estimated_seconds = len(df_players) * actual_rate
estimated_minutes = estimated_seconds / 60
estimated_hours = estimated_minutes / 60
print(f"Estimated time: {estimated_minutes:.0f} minutes ({estimated_hours:.1f} hours)")
print(f"Based on test run: 20 players in 3m 54.6s = {actual_rate:.1f}s per player")

# Define target stats
target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'ws', 'ws_per_48']

# Store all extracted data
all_player_stats = []
successful_extractions = 0
failed_extractions = 0
failed_players = []

# Track progress
start_time = datetime.now()
save_interval = 100  # Save progress every 100 players

print(f"\nStarting extraction at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("Progress will be saved every 100 players...")

for idx, (_, player) in enumerate(df_players.iterrows(), 1):
    player_name = player['player_name']
    player_url = player['player_url']

    # Progress indicator
    if idx % 50 == 0 or idx <= 10:
        elapsed = datetime.now() - start_time
        rate = idx / elapsed.total_seconds() * 60  # players per minute
        remaining = (len(df_players) - idx) / rate if rate > 0 else 0
        print(f"[{idx}/{len(df_players)}] {player_name} | Rate: {rate:.1f}/min | ETA: {remaining:.0f}min")

    try:
        # Navigate to player page
        driver.get(player_url)

        # Wait for advanced stats table
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "advanced"))
        )

        # Parse the page
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        advanced_table = soup.find('table', id='advanced')

        if advanced_table:
            # Get data rows (exclude headers and totals)
            rows = advanced_table.find('tbody').find_all('tr')
            data_rows = []

            for row in rows:
                season_cell = row.find('th', {'data-stat': 'year_id'})
                if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
                    data_rows.append(row)

            # Extract stats for each season
            for row in data_rows:
                season_data = {'player_name': player_name}

                for stat in target_stats:
                    cell = row.find(['th', 'td'], {'data-stat': stat})
                    if cell:
                        value = cell.text.strip()
                        season_data[stat] = value if value else None
                    else:
                        season_data[stat] = None

                all_player_stats.append(season_data)

            successful_extractions += 1

        else:
            failed_extractions += 1
            failed_players.append({'player': player_name, 'reason': 'No advanced table'})

    except Exception as e:
        failed_extractions += 1
        failed_players.append({'player': player_name, 'reason': str(e)})

    # Save progress every 100 players
    if idx % save_interval == 0:
        df_temp = pd.DataFrame(all_player_stats)
        df_temp.to_csv(f'../data/pages/nba_stats_progress_{idx}.csv', index=False)
        print(f"  -> Progress saved: {len(all_player_stats)} seasons extracted")

    # Rate limiting - 3 second delay
    time.sleep(3)

# Final save
df_final = pd.DataFrame(all_player_stats)
df_final.to_csv('../data/pages/nba_all_player_stats.csv', index=False)

# Save failed players log
if failed_players:
    df_failed = pd.DataFrame(failed_players)
    df_failed.to_csv('../data/pages/failed_extractions.csv', index=False)

# Final summary
end_time = datetime.now()
total_time = end_time - start_time

print(f"\n=== EXTRACTION COMPLETE ===")
print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total time: {total_time}")
print(f"Successful players: {successful_extractions}/{len(df_players)} ({successful_extractions/len(df_players)*100:.1f}%)")
print(f"Failed players: {failed_extractions}")
print(f"Total seasons extracted: {len(df_final)}")
print(f"Average seasons per player: {len(df_final)/successful_extractions:.1f}")
print(f"\nFinal dataset saved to: '../data/pages/nba_all_player_stats.csv'")

# Close browser
driver.quit()
print("Browser closed.")

Starting full dataset extraction...
Total players to process: 5313
Estimated time: 1039 minutes (17.3 hours)
Based on test run: 20 players in 3m 54.6s = 11.7s per player

Starting extraction at 2025-07-03 22:55:21
Progress will be saved every 100 players...
[1/5313] Alaa Abdelnaby | Rate: 38387.7/min | ETA: 0min
[2/5313] Zaid Abdul-Aziz | Rate: 7.4/min | ETA: 718min
[3/5313] Kareem Abdul-Jabbar | Rate: 7.6/min | ETA: 702min
[4/5313] Mahmoud Abdul-Rauf | Rate: 2.0/min | ETA: 2601min
[5/5313] Tariq Abdul-Wahad | Rate: 2.4/min | ETA: 2208min
[6/5313] Shareef Abdur-Rahim | Rate: 2.7/min | ETA: 1945min
[7/5313] Tom Abernethy | Rate: 3.0/min | ETA: 1767min
[8/5313] Forest Able | Rate: 3.3/min | ETA: 1591min
[9/5313] John Abramovic | Rate: 3.6/min | ETA: 1454min
[10/5313] Álex Abrines | Rate: 4.0/min | ETA: 1342min
[50/5313] Cory Alexander | Rate: 8.6/min | ETA: 615min
[100/5313] J.J. Anderson | Rate: 9.4/min | ETA: 556min
  -> Progress saved: 688 seasons extracted
[150/5313] Carlos Arroyo | 

In [24]:
# Force fresh ChromeDriver download and setup
import shutil

print("Forcing fresh ChromeDriver download...")

# Remove the cached ChromeDriver
cache_path = "/Users/jankasen/.wdm/drivers/chromedriver"
try:
    if os.path.exists(cache_path):
        shutil.rmtree(cache_path)
        print("✓ Removed cached ChromeDriver")
    else:
        print("No cached ChromeDriver found")
except Exception as e:
    print(f"Could not remove cache: {e}")

# Try different approach - let ChromeDriverManager download fresh version
print("\nDownloading fresh ChromeDriver...")

try:
    # Force fresh download
    from webdriver_manager.chrome import ChromeDriverManager
    from webdriver_manager.core.os_manager import ChromeType
    
    # Get fresh driver path
    driver_path = ChromeDriverManager().install()
    print(f"✓ Fresh ChromeDriver downloaded to: {driver_path}")
    
    # Test if the driver executable works
    import subprocess
    result = subprocess.run([driver_path, "--version"], 
                          capture_output=True, text=True, timeout=10)
    
    if result.returncode == 0:
        print(f"✓ ChromeDriver version: {result.stdout.strip()}")
        
        # Now try to create browser
        print("\nCreating browser with fresh driver...")
        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        service = ChromeService(driver_path)
        driver = webdriver.Chrome(service=service, options=options)
        
        # Test browser
        driver.get("https://www.basketball-reference.com")
        print(f"✓ Browser working: {driver.title}")
        
    else:
        print(f"✗ ChromeDriver test failed: {result.stderr}")
        
except Exception as e:
    print(f"✗ Setup failed: {e}")
    
    # Last resort - try system Chrome if available
    print("\nTrying system Chrome as last resort...")
    try:
        # On Mac, try using system Chrome
        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.binary_location = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
        
        driver = webdriver.Chrome(options=options)
        driver.get("https://www.basketball-reference.com")
        print(f"✓ System Chrome working: {driver.title}")
        
    except Exception as e2:
        print(f"✗ System Chrome also failed: {e2}")
        print("\nTroubleshooting suggestions:")
        print("1. Restart Jupyter notebook completely")
        print("2. Update Chrome browser: Chrome → About Google Chrome")
        print("3. Try: pip install --upgrade selenium webdriver-manager")
        print("4. Restart your computer if all else fails")

Forcing fresh ChromeDriver download...
✓ Removed cached ChromeDriver

✓ Fresh ChromeDriver downloaded to: /Users/jankasen/.wdm/drivers/chromedriver/mac64/138.0.7204.92/chromedriver-mac-arm64/chromedriver
✓ ChromeDriver version: ChromeDriver 138.0.7204.92 (f079b9bc781e3c2adb1496ea1d72812deb0ddb3d-refs/branch-heads/7204_50@{#8})

Creating browser with fresh driver...


Service process refused to terminate gracefully with SIGTERM, escalating to SIGKILL.
Traceback (most recent call last):
  File "/opt/anaconda3/envs/boardman/lib/python3.11/site-packages/selenium/webdriver/common/service.py", line 177, in _terminate_process
    self.process.wait(60)
  File "/opt/anaconda3/envs/boardman/lib/python3.11/subprocess.py", line 1264, in wait
    return self._wait(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/boardman/lib/python3.11/subprocess.py", line 2045, in _wait
    raise TimeoutExpired(self.args, timeout)
subprocess.TimeoutExpired: Command '['/Users/jankasen/.wdm/drivers/chromedriver/mac64/138.0.7204.92/chromedriver-mac-arm64/chromedriver', '--port=54857']' timed out after 60 seconds


✓ Browser working: Basketball Statistics & History of Every Team & NBA and WNBA Players | Basketball-Reference.com


In [29]:
# Recovery extraction for remaining players
print("Starting recovery extraction...")

# Load original player database
print("Loading original player database...")
df_players = pd.read_csv('../data/01-pages/all_nba_players.csv')
print(f"Loaded {len(df_players)} total players")

# Load existing data to identify what's already extracted
try:
    # Load from the most recent progress file that has data
    df_extracted = pd.read_csv('../data/02-player-stats-extraction/extraction_run_1/nba_stats_progress_1400.csv')
    extracted_players = set(df_extracted['player_name'].unique())
    print(f"Found {len(extracted_players)} already extracted players")
except FileNotFoundError:
    extracted_players = set()
    print("No existing extraction file found")

# Identify remaining players
remaining_players = df_players[~df_players['player_name'].isin(extracted_players)]
print(f"Players still needed: {len(remaining_players)}")

if len(remaining_players) == 0:
    print("All players already extracted!")
else:
    # Show remaining letters distribution
    print("\nRemaining players by letter:")
    print(remaining_players['letter'].value_counts().sort_index())

    # Setup for extraction with browser restarts
    target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'ws', 'ws_per_48']
    all_new_stats = []
    successful_extractions = 0
    failed_extractions = 0
    failed_players = []
    
    # Browser restart interval
    restart_interval = 200
    save_interval = 50
    
    def setup_browser():
        """Setup fresh browser instance with simpler configuration"""
        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        service = ChromeService(ChromeDriverManager().install())
        return webdriver.Chrome(service=service, options=options)
    
    # Check if we already have a working driver from previous cells
    try:
        # Test if existing driver is still working
        driver.current_url
        print("Using existing browser instance...")
    except:
        # Create new browser if needed
        print("Setting up new browser instance...")
        driver = setup_browser()
    start_time = datetime.now()
    
    print(f"\nStarting recovery at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    try:
        for idx, (_, player) in enumerate(remaining_players.iterrows(), 1):
            player_name = player['player_name']
            player_url = player['player_url']
            
            # Restart browser every 200 players to prevent memory issues
            if idx % restart_interval == 0:
                print(f"\n  Restarting browser at player {idx}...")
                driver.quit()
                time.sleep(5)  # Brief pause
                driver = setup_browser()
            
            # Progress indicator
            if idx % 25 == 0 or idx <= 10:
                elapsed = datetime.now() - start_time
                rate = idx / elapsed.total_seconds() * 60 if elapsed.total_seconds() > 0 else 0
                remaining_time = (len(remaining_players) - idx) / rate if rate > 0 else 0
                print(f"[{idx}/{len(remaining_players)}] {player_name} | Rate: {rate:.1f}/min | ETA: {remaining_time:.0f}min")
            
            try:
                # Navigate to player page
                driver.get(player_url)
                
                # Wait for advanced stats table with longer timeout
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.ID, "advanced"))
                )
                
                # Parse the page
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                advanced_table = soup.find('table', id='advanced')
                
                if advanced_table:
                    # Get data rows
                    rows = advanced_table.find('tbody').find_all('tr')
                    data_rows = []
                    
                    for row in rows:
                        season_cell = row.find('th', {'data-stat': 'year_id'})
                        if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
                            data_rows.append(row)
                    
                    # Extract stats for each season
                    for row in data_rows:
                        season_data = {'player_name': player_name}
                        
                        for stat in target_stats:
                            cell = row.find(['th', 'td'], {'data-stat': stat})
                            if cell:
                                value = cell.text.strip()
                                season_data[stat] = value if value else None
                            else:
                                season_data[stat] = None
                        
                        all_new_stats.append(season_data)
                    
                    successful_extractions += 1
                    
                else:
                    failed_extractions += 1
                    failed_players.append({'player': player_name, 'reason': 'No advanced table'})
            
            except TimeoutException:
                failed_extractions += 1
                failed_players.append({'player': player_name, 'reason': 'Timeout waiting for table'})
            except WebDriverException as e:
                failed_extractions += 1
                failed_players.append({'player': player_name, 'reason': f'WebDriver error: {str(e)[:100]}'})
            except Exception as e:
                failed_extractions += 1
                failed_players.append({'player': player_name, 'reason': f'Unknown error: {str(e)[:100]}'})
            
            # Save progress more frequently
            if idx % save_interval == 0:
                if all_new_stats:
                    df_new = pd.DataFrame(all_new_stats)
                    df_new.to_csv(f'../data/02-player-stats-extraction/extraction_run_2/recovery_progress_{idx}.csv', index=False)
                    print(f"  -> Recovery progress saved: {len(all_new_stats)} new seasons")
            
            # Rate limiting
            time.sleep(3)
    
    finally:
        driver.quit()
    
    # Combine with existing data and save
    if all_new_stats:
        df_new = pd.DataFrame(all_new_stats)
        
        # Combine with existing data
        if 'df_extracted' in locals():
            df_combined = pd.concat([df_extracted, df_new], ignore_index=True)
        else:
            df_combined = df_new
        
        # Save combined dataset
        df_combined.to_csv('../data/02-player-stats-extraction/nba_all_player_stats_complete.csv', index=False)
        
        # Save new failed players
        if failed_players:
            df_failed_new = pd.DataFrame(failed_players)
            df_failed_new.to_csv('../data/02-player-stats-extraction/extraction_run_2/recovery_failed_extractions.csv', index=False)
    
    # Final summary
    end_time = datetime.now()
    total_time = end_time - start_time
    
    print(f"\n=== RECOVERY COMPLETE ===")
    print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total time: {total_time}")
    print(f"New successful extractions: {successful_extractions}/{len(remaining_players)}")
    print(f"New failed extractions: {failed_extractions}")
    print(f"New seasons extracted: {len(all_new_stats)}")
    
    if 'df_combined' in locals():
        total_unique_players = len(df_combined['player_name'].unique())
        total_seasons = len(df_combined)
        print(f"\nFinal dataset:")
        print(f"Total unique players: {total_unique_players}")
        print(f"Total seasons: {total_seasons}")
        print(f"Average seasons per player: {total_seasons/total_unique_players:.1f}")
        print(f"Saved to: '../data/02-player-stats-extraction/nba_all_player_stats_complete.csv'")

Starting recovery extraction...
Loading original player database...
Loaded 5313 total players
Found 1383 already extracted players
Players still needed: 3921

Remaining players by letter:
letter
E      9
F    161
G    265
H    376
I     29
J    263
K    183
L    209
M    502
N    112
O     97
P    238
Q      9
R    267
S    464
T    209
U     12
V     61
W    413
Y     22
Z     20
Name: count, dtype: int64
Setting up new browser instance...


KeyboardInterrupt: 

In [30]:
# Slow recovery extraction with extended delays to avoid rate limiting
print("Starting slow recovery extraction with extended delays...")

# Load original player database
print("Loading original player database...")
df_players = pd.read_csv('../data/01-pages/all_nba_players.csv')
print(f"Loaded {len(df_players)} total players")

# Load existing data from BOTH extraction runs
extracted_players = set()

# Load from run 1
try:
    df_run1 = pd.read_csv('../data/02-player-stats-extraction/extraction_run_1/nba_stats_progress_1400.csv')
    extracted_players.update(df_run1['player_name'].unique())
    print(f"Found {len(df_run1['player_name'].unique())} players from extraction run 1")
except FileNotFoundError:
    print("No run 1 data found")

# Load from run 2 (latest progress file)
try:
    df_run2 = pd.read_csv('../data/02-player-stats-extraction/extraction_run_2/recovery_progress_1850.csv')
    extracted_players.update(df_run2['player_name'].unique())
    print(f"Found {len(df_run2['player_name'].unique())} players from extraction run 2")

    # Combine both runs for final dataset
    all_existing_data = pd.concat([df_run1, df_run2], ignore_index=True) if 'df_run1' in locals() else df_run2
    print(f"Total existing seasons: {len(all_existing_data)}")
except FileNotFoundError:
    print("No run 2 data found")
    all_existing_data = df_run1 if 'df_run1' in locals() else pd.DataFrame()

print(f"Total unique players already extracted: {len(extracted_players)}")

# Identify remaining players
remaining_players = df_players[~df_players['player_name'].isin(extracted_players)]
print(f"Players still needed: {len(remaining_players)}")

if len(remaining_players) == 0:
    print("All players already extracted!")
else:
    print(f"\nRemaining players by letter:")
    print(remaining_players['letter'].value_counts().sort_index())

    # Setup for slow extraction
    target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'ws', 'ws_per_48']
    all_new_stats = []
    successful_extractions = 0
    failed_extractions = 0
    failed_players = []

    # MUCH MORE CONSERVATIVE SETTINGS
    restart_interval = 100  # Restart browser more frequently
    save_interval = 25      # Save progress more frequently
    base_delay = 8          # Longer base delay between requests
    long_break_interval = 50 # Take longer breaks more often

    def setup_browser():
        """Setup browser with more conservative options"""
        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-extensions')
        # Add user agent to appear more like regular browser
        options.add_argument('--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36')
        service = ChromeService(ChromeDriverManager().install())
        return webdriver.Chrome(service=service, options=options)

    # Setup fresh browser
    print("Setting up conservative browser...")
    driver = setup_browser()
    start_time = datetime.now()

    print(f"\nStarting slow extraction at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Using {base_delay}s delays, {long_break_interval}-player break intervals")

    try:
        for idx, (_, player) in enumerate(remaining_players.iterrows(), 1):
            player_name = player['player_name']
            player_url = player['player_url']

            # Restart browser every 100 players
            if idx % restart_interval == 0:
                print(f"\n  Restarting browser at player {idx}...")
                driver.quit()
                time.sleep(10)  # Longer pause between restarts
                driver = setup_browser()

            # Take longer breaks every 50 players
            if idx % long_break_interval == 0:
                print(f"\n  Taking 2-minute break at player {idx}...")
                time.sleep(120)  # 2 minute break

            # Progress indicator (less frequent to reduce noise)
            if idx % 10 == 0 or idx <= 5:
                elapsed = datetime.now() - start_time
                rate = idx / elapsed.total_seconds() * 60 if elapsed.total_seconds() > 0 else 0
                remaining_time = (len(remaining_players) - idx) / rate if rate > 0 else 0
                print(f"[{idx}/{len(remaining_players)}] {player_name} | Rate: {rate:.1f}/min | ETA: {remaining_time:.0f}min")

            try:
                # Navigate to player page
                driver.get(player_url)

                # Wait longer for advanced stats table
                WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.ID, "advanced"))
                )

                # Parse the page
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                advanced_table = soup.find('table', id='advanced')

                if advanced_table:
                    # Get data rows
                    rows = advanced_table.find('tbody').find_all('tr')
                    data_rows = []

                    for row in rows:
                        season_cell = row.find('th', {'data-stat': 'year_id'})
                        if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
                            data_rows.append(row)

                    # Extract stats for each season
                    for row in data_rows:
                        season_data = {'player_name': player_name}

                        for stat in target_stats:
                            cell = row.find(['th', 'td'], {'data-stat': stat})
                            if cell:
                                value = cell.text.strip()
                                season_data[stat] = value if value else None
                            else:
                                season_data[stat] = None

                        all_new_stats.append(season_data)

                    successful_extractions += 1

                else:
                    failed_extractions += 1
                    failed_players.append({'player': player_name, 'reason': 'No advanced table'})

            except TimeoutException:
                failed_extractions += 1
                failed_players.append({'player': player_name, 'reason': 'Timeout waiting for table'})
                print(f"  ✗ Timeout: {player_name}")
            except WebDriverException as e:
                failed_extractions += 1
                failed_players.append({'player': player_name, 'reason': f'WebDriver error: {str(e)[:100]}'})
                print(f"  ✗ WebDriver error: {player_name}")
            except Exception as e:
                failed_extractions += 1
                failed_players.append({'player': player_name, 'reason': f'Unknown error: {str(e)[:100]}'})
                print(f"  ✗ Unknown error: {player_name}")

            # Save progress more frequently
            if idx % save_interval == 0:
                if all_new_stats:
                    df_new = pd.DataFrame(all_new_stats)
                    df_new.to_csv(f'../data/02-player-stats-extraction/extraction_run_3/slow_recovery_progress_{idx}.csv', index=False)
                    print(f"  -> Slow recovery progress saved: {len(all_new_stats)} new seasons")

            # LONGER rate limiting delay
            time.sleep(base_delay)

    finally:
        driver.quit()

    # Combine all data and save final dataset
    if all_new_stats:
        df_new_slow = pd.DataFrame(all_new_stats)

        # Combine with all existing data
        df_final_complete = pd.concat([all_existing_data, df_new_slow], ignore_index=True)

        # Save complete dataset
        df_final_complete.to_csv('../data/02-player-stats-extraction/nba_complete_player_stats.csv', index=False)

        # Save new failed players
        if failed_players:
            df_failed_slow = pd.DataFrame(failed_players)
            df_failed_slow.to_csv('../data/02-player-stats-extraction/extraction_run_3/slow_recovery_failed.csv', index=False)

    # Final summary
    end_time = datetime.now()
    total_time = end_time - start_time

    print(f"\n=== SLOW RECOVERY COMPLETE ===")
    print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total time: {total_time}")
    print(f"New successful extractions: {successful_extractions}/{len(remaining_players)}")
    print(f"New failed extractions: {failed_extractions}")
    print(f"New seasons extracted: {len(all_new_stats) if all_new_stats else 0}")
    
    if 'df_final_complete' in locals():
        total_unique_players = len(df_final_complete['player_name'].unique())
        total_seasons = len(df_final_complete)
        print(f"\nFinal complete dataset:")
        print(f"Total unique players: {total_unique_players}")
        print(f"Total seasons: {total_seasons}")
        print(f"Average seasons per player: {total_seasons/total_unique_players:.1f}")
        print(f"Saved to: '../data/02-player-stats-extraction/nba_complete_player_stats.csv'")

Starting slow recovery extraction with extended delays...
Loading original player database...
Loaded 5313 total players
Found 1383 players from extraction run 1
Found 1791 players from extraction run 2
Total existing seasons: 21697
Total unique players already extracted: 3174
Players still needed: 2111

Remaining players by letter:
letter
H      1
J      1
M    185
N    112
O     97
P    238
Q      9
R    267
S    464
T    209
U     12
V     61
W    413
Y     22
Z     20
Name: count, dtype: int64
Setting up conservative browser...

Starting slow extraction at 2025-07-04 18:35:41
Using 8s delays, 50-player break intervals
[1/2111] Don Hanrahan | Rate: 45558.1/min | ETA: 0min
[2/2111] Reggie Johnson | Rate: 1.4/min | ETA: 1539min
[3/2111] Darko Miličić | Rate: 1.7/min | ETA: 1227min
[4/2111] Nat Militzok | Rate: 2.1/min | ETA: 1005min
[5/2111] Andre Miller | Rate: 2.4/min | ETA: 867min
[10/2111] Brandon Miller | Rate: 2.6/min | ETA: 801min
[20/2111] Malcolm Miller | Rate: 2.7/min | ETA: 